# QAOA Max-Cut on a 27-node cycle

This notebook runs one QAOA layer (p=1) and samples candidate cuts. Choose **Runtime > Change runtime type > T4 GPU** for GPU execution. If Aer reports no GPU, it uses CPU with fewer shots. This is a simulator experiment, not a physical quantum computer.

[Project home](https://github.com/zistgah/tumtum) · [Zenodo DOI](https://doi.org/10.5281/zenodo.23861046) · [Algorithm source](https://github.com/zistgah/tumtum/tree/main/examples/qaoa-maxcut-27)

In [ ]:
%pip -q install qiskit-aer-gpu

In [ ]:
import math
from collections import defaultdict

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qubits = 27
edges = [(v, (v + 1) % qubits) for v in range(qubits)]
gamma = math.pi / 4
beta = math.pi / 8

circuit = QuantumCircuit(qubits)
circuit.h(range(qubits))
for left, right in edges:
    circuit.rzz(-gamma, left, right)
for qubit in range(qubits):
    circuit.rx(2 * beta, qubit)
circuit.measure_all()

devices = AerSimulator(method="statevector").available_devices()
device = "GPU" if "GPU" in devices else "CPU"
shots = 8192 if device == "GPU" else 1024
simulator = AerSimulator(method="statevector", device=device)
compiled = transpile(circuit, simulator, optimization_level=1)
result = simulator.run(compiled, shots=shots, seed_simulator=42).result()
if not result.success:
    raise RuntimeError(f"Aer run failed: {result.status}")

distribution = defaultdict(int)
for bitstring, count in result.get_counts(compiled).items():
    bits = bitstring.replace(" ", "")[::-1]
    cut = sum(bits[left] != bits[right] for left, right in edges)
    distribution[cut] += count

mean_cut = sum(cut * count for cut, count in distribution.items()) / shots
variance = sum(count * (cut - mean_cut) ** 2 for cut, count in distribution.items()) / shots
optimum = qubits - 1
print(f"Aer devices: {devices}; selected device: {device}; shots: {shots}")
print(f"QAOA p=1 mean cut: {mean_cut:.3f} / {qubits}")
print(f"Random-assignment baseline: {qubits / 2:.3f} / {qubits}")
print(f"Mean approximation ratio vs exact cycle optimum ({optimum}): {mean_cut / optimum:.3%}")
print(f"Best sampled cut: {max(distribution)} / {qubits} ({distribution[max(distribution)]} samples)")
print(f"Sampling standard error: {math.sqrt(variance / shots):.3f}")